# Experimente

API-Anbindung und Durchführung der Experimente. Voraussetzung: `llm_client.py` mit `ask_openai`, `ask_google` und `frage` liegt im selben ORdner; `.env` enthält `OPEN_API_KEY` und `GOOGLE_API_KEY`.

_Kostenregel_: Mechanik immer mit Mini-Prompts testen. Vollen Datensatz (~115.000 Tokens) nur im finalen, geplanten Lauf.


## 1. Setup


In [ ]:
import os, datetime
from dotenv import load_dotenv
from llm_client import ask_openai, ask_google, ask

# beim Entwickeln praktisch: llm_client.py wird bei Änderung neu geladen
%load_ext autoreload
%autoreload 2

load_dotenv() # Look for a .env file in the same directory as the Python script


sk-proj
AQ.Ab8R
=== OpenAI ===
antwort: funktioniert
version: gpt-5.6-luna
tokens: 16 / 5

=== Google ===
antwort: funktioniert
version: gemini-3.6-flash
tokens: 11 / 2



True

In [2]:
# Prüfen, dass beide Keys geladen sind 

print(os.environ.get('OPENAI_API_KEY', 'NICHT GEFUNDEN')[:7])
print(os.environ.get('GOOGLE_API_KEY', 'NICHT GEFUNDEN')[:7])

sk-proj
AQ.Ab8R


## 2. Modelle

Die konkreten Modellstrings zentral festlegen, damit sie nur an einer Stelle stehen.


In [4]:
MODELL_OPENAI = 'gpt-5.6-luna'
MODELL_GOOGLE = 'gemini-3.6-flash'

## 3. Verbindungstest


In [7]:
for name, anbieter, modell in [('OpenAI', 'openai', MODELL_OPENAI),
                               ('Google', 'google', MODELL_GOOGLE)]:
    print(f'=== {name} ===')
    try:
        r = ask(anbieter, 'Antworte mit genau einem Wort: funktioniert.', modell=modell)
        print('antwort:', r['antwort'])
        print('version:', r['modell_version'])
        print('tokens:', r['input_tokens'], '/',
            r['output_tokens'])
    except Exception as e:
        print('Fehler:', e)
    print()

=== OpenAI ===
antwort: funktioniert
version: gpt-5.6-luna
tokens: 16 / 5

=== Google ===
antwort: funktioniert
version: gemini-3.6-flash
tokens: 11 / 3



## 4. Datensatz laden

Der Rohdatensatz wird als CSV-Text vorbereitet, der später in datenbasierten Prompts eingefügt wird. Prompts mit diesem Text kosten je aufruf ~115.000 Input Tokens. Deshalb nur im geplanten Lauf verwenden, nicht zum Testen.


In [8]:
import pandas as pd
from pathlib import Path

df = pd.read_csv(Path('data') / 'ibm_original.csv')
data_as_text = df.to_csv(index=False)

print('Zeilen', len(df), '| Spalten:', df.shape[1])

Zeilen 1470 | Spalten: 35


# 5. Experiment-Runner

Iteration über Fragen x Modelle x Wiederholungen und speichert jede Antwort sofort als JSONL.


In [ ]:
# ## ACHTUNG TEURER TEST!!!

# import pandas as pd
# import json
# from pathlib import Path

# df = pd.read_csv(Path('data') / 'ibm_original.csv')

# # Datensatz als Text für den Prompt - CSV-Format ist kompakter als Markdown

# daten_als_text = df.to_csv(index=False)

# prompt = f"""Hier ist ein Datensatz im CSV-Format:

# {daten_als_text}

# Frage: Wie viele Zeilen (Beobachtungen) enthält der Datensatz genau?
# Antworte nur mit der Zahl."""

# r = ask('google', prompt)
# print("Antwort:", r['antwort'])
# print("Input-Tokens", r['input_tokens'])